In [2]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import os
import gurobipy as gp
import time
from gurobipy import GRB

# ====================================================
# 1. DATA INGESTION & UNBIASED SAMPLE LOCK
# ====================================================
input_file = r"E:\Github-assessment\Master-Project\Prescriptive\XGBoost_prescriptive_optimizer_inputs.csv"
print(f"[Integrated MILP V5] Loading risk fuel vectors: {input_file}")

if not os.path.exists(input_file):
    raise FileNotFoundError(input_file)

df_fuel = pd.read_csv(input_file, encoding='utf-8')

# Lock an unbiased random sample cohort to preserve experimental consistency
df_day = df_fuel.sample(n=12, random_state=42).reset_index(drop=True)

N_CASES = len(df_day)
N_ROOMS = 3
ROOM_CAPACITY = 480.0
BASE_TURNOVER = 20.0  # Statutory cleanup buffer between consecutive cases
DAY_END = 1440.0      # Hard physical 24-hour temporal wall

I = range(N_CASES)
R = range(N_ROOMS)

P50 = df_day["DURATION_P50_MINS"].values
P90 = df_day["DURATION_P90_MINS"].values

print(f" Integrated Pipeline Loaded: {N_CASES} cases mapped across {N_ROOMS} concurrent OR rooms.")

# ====================================================
# 2. PARAMETRIC PARETO SENSITIVITY WORKSPACE
# ====================================================
lambda_list = [0.0, 0.3, 0.5, 1.0]
frontier_results = []
gantt_registry = []

for lam in lambda_list:
    print(f"\n================ Optimization Frontier Lambda = {lam} ================\n")
    
    # Non-parametric robust duration invariant allocation
    duration = P50 + lam * (P90 - P50)
    duration = np.minimum(duration, 360.0) # Core cap filtration guardrail
    
    # SOLVER-FRIENDLY REFORMULATION: Tightened Big-M boundary bound 
    # M strictly bound to total workload sums + maximum possible turnovers to sharpen LP relaxation
    M = float(np.sum(duration) + (N_CASES * BASE_TURNOVER))

    # ====================================================
    # 3. ADVANCED DISJUNCTIVE MILP INITIALIZATION
    # ====================================================
    m = gp.Model(f"True_Disjunctive_OR_lambda_{lam}")
    m.setParam("OutputFlag", 0)
    m.setParam("MIPGap", 0.01)       # Journal-standard 1% convergence threshold
    m.setParam("TimeLimit", 60)      # 1-minute computing wall
    m.setParam("Heuristics", 0.4)    # Balanced heuristic feasibility pump
    m.setParam("Threads", 12)        # Prevent thread synchronization multi-core friction
    
    # Bounding variables
    x = m.addVars(I, R, vtype=GRB.BINARY, name="x_assign")
    s = m.addVars(I, lb=0.0, ub=DAY_END, vtype=GRB.CONTINUOUS, name="s_start")
    
    # Endogenous Completion tracking vector per unique case
    c_case = m.addVars(I, lb=0.0, vtype=GRB.CONTINUOUS, name="c_case")
    
    # Symmetrical Precedence Matrix variables restricted under i < j
    y = {}
    for i in I:
        for j in I:
            if i < j:
                for r in R:
                    y[i, j, r] = m.addVar(vtype=GRB.BINARY, name=f"y_order_{i}_{j}_{r}")
                    
    # Institutional room clock metrics
    C = m.addVars(R, lb=0.0, vtype=GRB.CONTINUOUS, name="C_room")
    OT = m.addVars(R, lb=0.0, vtype=GRB.CONTINUOUS, name="OT")
    IDLE = m.addVars(R, lb=0.0, vtype=GRB.CONTINUOUS, name="IDLE")
    
    # Minimax Chebyshev system metrics
    system_makespan = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="system_makespan")
    min_room_load = m.addVar(vtype=GRB.CONTINUOUS, lb=0.0, name="min_room_load")

    # ====================================================
    # 4. CAPACITY & LOGICAL COUPLING INTEGRATION
    # ====================================================
    
    # Constraint 1: Absolute single-room partitioning contract
    for i in I:
        m.addConstr(gp.quicksum(x[i, r] for r in R) == 1, name=f"Partition_{i}")

    # Constraint 2: Symmetrical Logical Implication Bounding Matrix (Tight MIP Cuts)
    for i in I:
        for j in I:
            if i < j:
                for r in R:
                    m.addConstr(y[i, j, r] <= x[i, r], name=f"Link_UpSrc_{i}_{j}_{r}")
                    m.addConstr(y[i, j, r] <= x[j, r], name=f"Link_UpTgt_{i}_{j}_{r}")
                    m.addConstr(y[i, j, r] >= x[i, r] + x[j, r] - 1, name=f"Link_LowForce_{i}_{j}_{r}")

    # Constraint 3: ENDOGENOUS TURNOVER INTEGRATION
    # Symmetrical Pairwise Non-overlap Disjunctive Constraints incorporating conditional cleaning padding
    for i in I:
        for j in I:
            if i >= j:
                continue
            for r in R:
                # Track Case Precedence: If y[i,j,r]=1, Case j follows i. 
                m.addConstr(
                    s[i] + duration[i] + BASE_TURNOVER <= s[j] + M * (1 - y[i, j, r]) + M * (2 - (x[i, r] + x[j, r])),
                    name=f"Disjunctive_Forward_{i}_{j}_{r}"
                )
                # Track Case Precedence: If y[i,j,r]=0, Case i follows j.
                m.addConstr(
                    s[j] + duration[j] + BASE_TURNOVER <= s[i] + M * y[i, j, r] + M * (2 - (x[i, r] + x[j, r])),
                    name=f"Disjunctive_Backward_{i}_{j}_{r}"
                )

    # Constraint 4: Case End-Clock Identity Mapping
    for i in I:
        m.addConstr(c_case[i] == s[i] + duration[i], name=f"Case_Completion_{i}")

    # Constraint 5: NO-TURNOVER-AT-END-OF-DAY (Endogenous Room Tail Allocation)
    # The room final completion C[r] only needs to cover the pure end time of whomever finishes last.
    for r in R:
        for i in I:
            m.addConstr(C[r] >= c_case[i] - M * (1 - x[i, r]), name=f"OR_Suite_Clock_Cap_{i}_{r}")

    # Constraint 6: Minimax System Makespan and Load Balance Boundaries
    for r in R:
        m.addConstr(system_makespan >= C[r], name=f"Minimax_Makespan_Cap_{r}")
        m.addConstr(min_room_load <= C[r], name=f"Minimax_Load_Floor_{r}")

    # Constraint 7: Institutional labor shift balances
    for r in R:
        m.addConstr(OT[r] - IDLE[r] == C[r] - ROOM_CAPACITY, name=f"Labor_Shift_Balance_{r}")

    # ====================================================
    # 5. WEIGHTED MULTI-CRITERIA PARETO OBJECTIVE
    # ====================================================
    objective_expression = (
        gp.quicksum(OT[r] * 1.00 for r in R) +
        gp.quicksum(IDLE[r] * 0.25 for r in R) +
        system_makespan * 0.50 +
        (system_makespan - min_room_load) * 0.10
    )
    
    m.setObjective(objective_expression, GRB.MINIMIZE)

    # ====================================================
    # 6. EX-POST EVALUATION PIPELINE
    # ====================================================
    print("   [MIP Engine] Unweaving dual-space mixed integer constraints...")
    start = time.time()

    m.optimize()

    solve_time = time.time() - start

    if m.SolCount > 0:
        obj_val = m.objVal
        overtime_mins = sum(OT[r].X for r in R)
        idle_mins = sum(IDLE[r].X for r in R)
        makespan_val = system_makespan.X
        load_gap_val = system_makespan.X - min_room_load.X
        
        # Invariant baseline operations metrics
        capacity_utilization = P50.sum() / (N_ROOMS * ROOM_CAPACITY)
        schedule_utilization = P50.sum() / sum(C[r].X for r in R)
        buffer_ratio = duration.sum() / P50.sum()
        
        # 解耦双向度量指标优化
        # 1. OnTime_Rate: 离散硬红线按时关台率 (如遭遇大盘超载，诚实汇报 0.0%)
        ontime_rate = sum(1 for r in R if C[r].X <= ROOM_CAPACITY) / N_ROOMS
        
        # 2. Cap_Expansion_Ratio: 连续容量被迫扩张率/超负荷指数 (完美捕捉随着 Lambda 注入的防御名义预算变化)
        total_statutory_capacity = N_ROOMS * ROOM_CAPACITY
        total_allocated_schedule = sum(C[r].X for r in R)
        capacity_expansion_ratio = total_allocated_schedule / total_statutory_capacity
        
        frontier_results.append({
            "Lambda": lam, 
            "Objective": obj_val, 
            "Overtime": overtime_mins, 
            "Idle": idle_mins,
            "Makespan": makespan_val, 
            "Cap_Util": capacity_utilization, 
            "Sched_Util": schedule_utilization,
            "Buffer_Ratio": buffer_ratio, 
            "Load_Gap": load_gap_val, 
            "OnTime_Rate": ontime_rate,
            "Cap_Expansion": capacity_expansion_ratio,
            "Solve_Time_sec": solve_time
        })
        
        print(f" Success! Obj: {obj_val:.2f} | OT: {overtime_mins:.1f}m | Makespan: {makespan_val:.1f}m | Sched Util: {schedule_utilization*100:.1f}%")
        
        # ====================================================
        # 7. GANTT-CHART NATIVE EXPORT REGISTRY
        # ====================================================
        for r in R:
            room_cases = []
            for i in I:
                if x[i, r].X > 0.5:
                    room_cases.append((s[i].X, i, duration[i], c_case[i].X))
            
            # Sort chronologically inside each parallel machine securely
            room_cases.sort()
            
            for slot_idx, (start_clock, case_idx, pure_dur, end_clock) in enumerate(room_cases, start=1):
                log_id = df_day["LOG_ID"].iloc[case_idx]
                
                # Factual Endogenous Turnover validation: End-of-day cancellation logic
                has_turnover_following = 1 if slot_idx < len(room_cases) else 0
                turnover_allocated = BASE_TURNOVER if has_turnover_following == 1 else 0.0
                
                gantt_registry.append({
                    "Lambda": lam,
                    "Room_ID": r + 1,
                    "Sequence_Slot": slot_idx,
                    "Patient_Log_ID": log_id,
                    "Case_Index": case_idx,
                    "Scheduled_Start_Min": start_clock,
                    "Pure_Surgical_End_Min": end_clock,
                    "Endogenous_Turnover_Paid": turnover_allocated,
                    "Total_Block_Release_Min": end_clock + turnover_allocated
                })
    else:
        print(f" Lambda={lam} failed convergence inside the tightened disjunctive hull.")

# ====================================================
# 8. STRUCTURAL DATABASE PERSISTENCE
# ====================================================
os.makedirs("Results", exist_ok=True)

frontier_df = pd.DataFrame(frontier_results)
gantt_df = pd.DataFrame(gantt_registry)

output_dir = "XGBoost_Results"
os.makedirs(output_dir, exist_ok=True)

summary_file = os.path.join(output_dir, "true_disjunctive_or_summary.csv")
gantt_file = os.path.join(output_dir, "true_disjunctive_or_gantt_schedule.csv")

frontier_df.to_csv(summary_file, index=False, encoding='utf-8')
gantt_df.to_csv(gantt_file, index=False, encoding='utf-8')

print("\n" + "="*110)
print("     FINAL PROSPECTIVE INTEGRATED PRECIPITATION FRONTIER COMPLETE (V5-DYNAMIC)")
print("="*110)
print(frontier_df.to_string(index=False, formatters={
    'Objective': '{:,.2f}'.format, 'Overtime': '{:,.2f}'.format, 'Idle': '{:,.2f}'.format,
    'Makespan': '{:,.1f}m'.format, 'Cap_Util': '{:.2%}'.format, 'Sched_Util': '{:.2%}'.format,
    'Buffer_Ratio': '{:.2f}x'.format, 'Load_Gap': '{:,.1f}m'.format, 
    'OnTime_Rate': '{:.1%}'.format, 'Cap_Expansion': '{:.2%}'.format, 'Solve_Time_sec': '{:.2f}s'.format
}))
print("="*110 + "\n")
print(f" SUCCESS! High-Fidelity Frontier Summary Logged  : {summary_file}")
print(f" SUCCESS! Gantt-Chart Native Matrix Exported       : {gantt_file}")

[Integrated MILP V5] Loading risk fuel vectors: E:\Github-assessment\Master-Project\Prescriptive\XGBoost_prescriptive_optimizer_inputs.csv
 Integrated Pipeline Loaded: 12 cases mapped across 3 concurrent OR rooms.

================ Optimization Frontier Lambda = 0.0 ================

   [MIP Engine] Unweaving dual-space mixed integer constraints...
 Success! Obj: 1073.73 | OT: 702.7m | Makespan: 735.6m | Sched Util: 91.6%

================ Optimization Frontier Lambda = 0.3 ================

   [MIP Engine] Unweaving dual-space mixed integer constraints...
 Success! Obj: 1752.76 | OT: 1278.6m | Makespan: 938.5m | Sched Util: 72.2%

================ Optimization Frontier Lambda = 0.5 ================

   [MIP Engine] Unweaving dual-space mixed integer constraints...
 Success! Obj: 2150.56 | OT: 1628.2m | Makespan: 1039.6m | Sched Util: 64.0%

================ Optimization Frontier Lambda = 1.0 ================

   [MIP Engine] Unweaving dual-space mixed integer constraints...
 Success! 